Here just show the case of sc-SVC, refer to our [Reproduced notebook dir](./reproduce/case) for further cases in real applications.

In [ ]:
import os

output_dir = "output/sc_SVC_case"
raw_data_dir = "./raw_data/Real_application"

patient_id = "P2CRC"
data_type = "Xenium"
output_dir = f"{output_dir}/{patient_id}_{data_type}"
sc_ref_file="adata_sc_all_reanno.h5ad"

cell_type_col = "Level1"

In [ ]:
select_ct = "T"
output_dir = f"{output_dir}/{select_ct}"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
from revise.conf.application_sc_conf import ApplicationScConf

config = ApplicationScConf(
    sample_name=patient_id,
    raw_data_path=raw_data_dir,
    result_root_path=output_dir,
    cell_type_col=cell_type_col,
    confidence_col="Confidence",
    unknown_key="Unknown",
    st_file=f"{data_type}.h5ad",
    sc_ref_file=sc_ref_file
)

In [ ]:
import scanpy as sc

adata_st = sc.read_h5ad(config.st_file_path)

adata_sc_ref = sc.read_h5ad(config.sc_ref_file_path)
adata_sc_ref = adata_sc_ref[adata_sc_ref.obs['Patient'] == patient_id, :]

In [ ]:
from revise.methods.global_anchoring import GlobalAnchoring
from revise.application import ScSVC
from revise.tools.log import Logger

logger = Logger(name=f"run.log", log_file=f'{output_dir}/application_sc.log').get_logger()
sc_svc = ScSVC(adata_st, adata_sc_ref, config, logger)
annotate_method = GlobalAnchoring(config, logger)
adata_st = annotate_method.run(sc_svc.st_adata, sc_svc.sc_ref_adata, cell_type_col = "Level1")

In [ ]:
print(select_ct)
ct_adata_sc = sc_svc.sc_ref_adata[sc_svc.sc_ref_adata.obs[cell_type_col] == select_ct]
ct_adata_sp = adata_st[adata_st.obs[cell_type_col] == select_ct]

In [ ]:
subcell_type_col = "Level2"
ct_adata_sp = annotate_method.run(ct_adata_sp, ct_adata_sc, cell_type_col = subcell_type_col)

In [ ]:
from revise.methods.graph_cluster import GraphCluster

# resolutions = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
resolutions = [0.6, 0.7, 0.8]

graph_cluster = GraphCluster(config, logger)
sc_SVC_adata, merge_df, best_res = graph_cluster.run(ct_adata_sp, resolutions, subcell_type_col)

sc_SVC_adata

In [ ]:
sc_SVC_adata.obs['SVC_cluster'] = sc_SVC_adata.obs[f'leiden_{best_res}']
sp_cluster_num = merge_df.loc[merge_df['resolution'] == best_res, 'cluster_num'].values[0]
sp_cluster_num

In [ ]:
ct_adata_sc = annotate_method.run(ct_adata_sc, sc_SVC_adata, cell_type_col = "SVC_cluster")
ct_adata_sc

In [ ]:
sc_SVC_adata.write(f"{output_dir}/sc_SVC_spatial.h5ad")
ct_adata_sc.write(f"{output_dir}/sc_SVC_expr.h5ad")

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

def get_cm(adata, row, col):
    grouped = adata.obs.groupby([row, col]).size()
    cm_df = grouped.unstack(fill_value=0)
    
    return cm_df
def plot_cm(cm_df, save_dir=None):
    
    plt.figure(figsize=(10, 8))
    
    sns.heatmap(cm_df, 
                    annot=True, 
                    fmt='',
                    cmap='Reds',
                    )

    
    plt.title('Confusion Matrix')
    plt.xlabel('Expert anno')
    plt.ylabel('sc_SVC cluster')
    
    if save_dir is not None:
        plt.savefig(f"{save_dir}/cm.pdf", bbox_inches='tight')
        plt.close()
    else:
        plt.show()


In [ ]:
cm_df = get_cm(ct_adata_sc, 'SVC_cluster', subcell_type_col)
plot_cm(cm_df, save_dir = output_dir)
cm_df

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors 

size = None
# size = 20
cmap = plt.cm.get_cmap('tab20', lut=10
                  )
palette = [mcolors.to_hex(cmap(i)) for i in range(cmap.N)]

sc.pl.scatter(sc_SVC_adata, x="x", y="y", 
              color = 'Level2',
              # palette=palette,
            size = size
            )
desired_order = ['1','2','3','5','4','0'] # for better visualization
sc_SVC_adata.obs['SVC_cluster'] = (
    sc_SVC_adata.obs['SVC_cluster']
         .cat
         .reorder_categories(desired_order, ordered=True)   # ordered=True 可选
)

sc.pl.scatter(sc_SVC_adata, x="x", y="y", 
              color = 'SVC_cluster',
              # palette=palette,
            size = size
            )

In [ ]:
import matplotlib.pyplot as plt
def plot_sc_SVC(adata, color, title = None, file_name = None):

    plt.figure(figsize=(10, 8*len(color)))
    sc.pl.scatter(adata, x="x", y="y", 
            color = color,
            title=title, show = False,
            )
    plt.savefig(file_name, dpi = 300)
    plt.close()
    

sc_SVC_file_name = f"{output_dir}/sc_SVC.png"
plot_sc_SVC(sc_SVC_adata, color = 'SVC_cluster',
            file_name = sc_SVC_file_name
            )

sc_SVC_file_name = f"{output_dir}/compare_Level2.png"
plot_sc_SVC(sc_SVC_adata, color = 'Level2',
            title="Expert anno",
            file_name = sc_SVC_file_name
            )

plot_sc_SVC(sc_SVC_adata, color = 'SVC_cluster',
            title="sc_SVC",
            file_name = sc_SVC_file_name
            )
